# SynthesisSimilarity — interactive tour

One section per functionality of the PrecursorSelector codebase (He et al., *Sci. Adv.* 2023).
Run **Setup** once (~20–30 s: loads the model and embeds the whole knowledge base), then run any
section independently. Every section has its `# INPUT` variables at the top of the cell — edit and re-run.

| § | Functionality | Source to read alongside |
|---|---------------|--------------------------|
| 1 | composition → embedding | `core/encoders.py`, `core/model_framework.py::get_mat_vector` |
| 2 | target–target similarity | `scripts/_02_target_material_similarity.py` |
| 3 | knowledge-base retrieval (nearest known targets) | `scripts_utils/precursors_recommendation_utils.py::recommend_precursors` (first half) |
| 4 | recipes reported for a known target | `scripts_utils/recommendation_utils.py::collect_targets_in_reactions` |
| 5 | masked precursor completion (MPC) | `core/task_models.py::PrecursorsPredict`, `core/callbacks.py`, `scripts/_03` |
| 6 | full precursor recommendation | `scripts_utils/precursors_recommendation_utils.py`, `scripts/_01` |
| 7 | reaction balancing / validation | `ReactionCompleter`, `scripts_utils/reaction_utils.py` |
| 8 | material decoder (the `mat` task) | `core/task_models.py::MaterialDecoder`, `model_framework.py::recover_mat` |

Kernel: **SynthesisSimilarity (.venv)** (TF 2.15, the repo's patched environment).

In [1]:
# ---------- Setup: run once ----------
import numpy as np
import pandas as pd
from pprint import pprint

from SynthesisSimilarity import PrecursorsRecommendation
from SynthesisSimilarity.core.utils import formula_to_array, array_to_formula
from SynthesisSimilarity.core.mat_featurization import featurize_list_of_composition

rec = PrecursorsRecommendation()  # loads model + embeds all knowledge-base targets
model = rec.framework_model
ALL_ELES = rec.all_elements


def embed(formulas):
    # formula strings -> L2-normalized PrecursorSelector embeddings, shape (n, 32)
    comps = [formula_to_array(f, ALL_ELES) for f in formulas]
    feats = featurize_list_of_composition(
        comps=comps, ele_order=ALL_ELES, featurizer_type=rec.featurizer_type
    )
    vecs = model.get_mat_vector(np.array(feats)).numpy()
    return vecs / np.linalg.norm(vecs, axis=-1, keepdims=True)


print(
    f"knowledge base: {len(rec.train_targets_formulas)} unique targets, "
    f"embedding dim {rec.train_targets_vecs.shape[1]}, "
    f"{len(ALL_ELES)} elements in composition vector"
)

1/1 [==============================] - 1s 1s/step - loss: 0.0000e+00
local_data.keys() ['train_reactions', 'val_reactions', 'test_reactions']
len(train_targets) 28598
knowledge base: 28598 unique targets, embedding dim 32, 83 elements in composition vector


## 1 · Composition → embedding

A target is represented only by its **composition** (fraction per element, an ~80-dim vector).
The encoder maps it to a 32-dim vector. Everything else in the pipeline — similarity, retrieval,
MPC — operates on this vector. The encoder weights are what training shapes.

In [2]:
# INPUT: any chemical formulas
formulas = ["Li3NbO4", "NaNb3O8", "Li3TaO4"]

vecs = embed(formulas)
print("shape:", vecs.shape)
pd.DataFrame(vecs, index=formulas).round(3)

shape: (3, 32)


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
Li3NbO4,-0.061,0.113,0.020,-0.258,0.057,-0.011,0.173,-0.273,0.021,-0.008,...,0.140,0.106,-0.137,0.025,-0.065,0.098,0.168,-0.287,0.341,-0.041
NaNb3O8,-0.070,-0.197,0.215,-0.238,-0.018,-0.036,0.136,-0.274,0.073,0.128,...,0.021,0.256,-0.150,0.017,0.294,-0.057,0.082,-0.308,0.210,-0.179
Li3TaO4,-0.007,0.034,-0.078,-0.031,0.254,-0.098,0.157,-0.341,0.031,-0.028,...,-0.037,0.082,-0.133,-0.058,-0.150,0.121,0.175,-0.314,0.206,-0.018


## 2 · Target–target similarity

Cosine similarity of embeddings = **synthesis similarity**: the training objective pulls together
materials made from similar precursors, so this is *not* structural or compositional similarity.
Compare chemically related vs unrelated pairs and see how it differs from your intuition.

In [3]:
# INPUT: a set of formulas to compare pairwise
formulas = ["LiFePO4", "LiMn2O4", "LiNi0.5Mn1.5O4", "BaTiO3", "SrTiO3", "YBa2Cu3O7"]

vecs = embed(formulas)
pd.DataFrame(vecs @ vecs.T, index=formulas, columns=formulas).round(3)

,LiFePO4,LiMn2O4,LiNi0.5Mn1.5O4,BaTiO3,SrTiO3,YBa2Cu3O7
LiFePO4,1.000,0.643,0.640,0.304,0.465,0.289
LiMn2O4,0.643,1.000,0.936,0.338,0.483,0.283
LiNi0.5Mn1.5O4,0.640,0.936,1.000,0.305,0.494,0.178
BaTiO3,0.304,0.338,0.305,1.000,0.758,0.714
SrTiO3,0.465,0.483,0.494,0.758,1.000,0.541
YBa2Cu3O7,0.289,0.283,0.178,0.714,0.541,1.000


## 3 · Knowledge-base retrieval

The first half of recommendation: embed the (possibly novel) target, rank all known targets by
cosine similarity. The recommender then walks down this list borrowing precursor sets.
`#papers` = how many text-mined reactions report that target.

In [4]:
# INPUT
target = "LiFePO4"
k = 10

v = embed([target])
sims = (v @ rec.train_targets_vecs.T)[0]
top = np.argsort(sims)[::-1][:k]

rows = []
for i in top:
    recipe = rec.train_targets_recipes[i]
    best_pres, _ = recipe["pres"].most_common(1)[0]
    rows.append(
        {
            "known target": rec.train_targets_formulas[i],
            "similarity": round(float(sims[i]), 3),
            "#papers": sum(recipe["pres"].values()),
            "most common precursor set": " + ".join(best_pres),
        }
    )
pd.DataFrame(rows)

,known target,similarity,#papers,most common precursor set
0,LiFePO4,1.000,154,Fe(CO2)2 + Li2CO3 + PH6NO4
1,Li89240Fe84991(PO4)84991,1.000,1,Fe(CO2)2 + Li2CO3 + PH6NO4
2,Li9Fe10(PO4)10,1.000,1,Fe2O3 + Li3PO4 + PH9(NO2)2
3,Li4Fe5(PO4)5,1.000,1,Fe2O3 + Li3PO4 + PH9(NO2)2
4,Li3Fe2(PO4)3,0.999,7,Fe2O3 + Li2CO3 + PH6NO4
5,Li2FeP2O7,0.998,1,Fe + Li + PO4
6,LiFeP2O7,0.995,3,Fe2O3 + LiP(HO2)2 + PH6NO4
7,NaLi19Fe20(PO4)20,0.966,1,Fe(CO2)2 + Li2CO3 + Na2CO3 + PH6NO4
8,NaLi9Fe10(PO4)10,0.964,1,Fe(CO2)2 + Li2CO3 + Na2CO3 + PH6NO4
9,Li96TiFe100(PO4)100,0.961,1,Fe(CO2)2 + Li2CO3 + PH6NO4 + TiH12(CO)4


## 4 · Recipes reported for a known target

Direct lookup in the text-mined dataset (no ML): every distinct precursor set reported for a
target, with counts. This is also the **ground truth** the paper's benchmark scores against
(`_05_recommendation_benchmark.py`: success = an exact match of one proposed set against these keys).

In [5]:
# INPUT: a target present in the knowledge base (borrow one from section 3's output)
known_target = "BaTiO3"

recipe = rec.train_targets[known_target]
print("synthesis types:", dict(recipe["syn_type"]))
print("number of source reactions:", len(recipe["raw_index"]))
pd.DataFrame(
    [{"precursor set": " + ".join(p), "count": c} for p, c in recipe["pres"].most_common()]
)

synthesis types: {'solid_state_ceramic_synthesis': 392}
number of source reactions: 392


,precursor set,count
0,BaCO3 + TiO2,363
1,BaO + TiO2,13
2,BaO2 + TiO2,4
3,Ba(NO3)2 + TiO2,4
4,Ba(HO)2 + TiO2,3
5,BaCO3 + Ti2O3,2
6,BaCO3 + Ti,1
7,BaCO3 + CSN2 + TiO2,1
8,Ba2CO3 + TiO2,1


## 5 · Masked precursor completion (MPC)

The model's main training task: given the target and a *partial* precursor set, predict what's
missing. Try fixing an unusual first precursor (e.g. a nitrate instead of an oxide) and watch the
completion adapt — this conditioning is what makes recommended sets internally consistent.
Scores are the head's confidence per candidate precursor (knowledge-base precursor vocabulary).

In [6]:
# INPUT: target + already-chosen precursors (empty list = predict from scratch)
target = "LaAlO3"
partial_precursors = ["La(NO3)3"]

zero = np.zeros(len(ALL_ELES), dtype=np.float32)
cond = [formula_to_array(p, ALL_ELES) for p in partial_precursors]
cond += [zero] * (rec.max_mats_num - 1 - len(cond))

pre_lists, pre_str_lists = rec.predict_precursor_callback.predict_precursors(
    model,
    target_compositions=np.array([formula_to_array(target, ALL_ELES)]),
    precursors_conditional=np.array([cond]),
    to_print=False,
)
pd.DataFrame(pre_str_lists[0][:15], columns=["predicted precursor", "score"]).round(4)

,predicted precursor,score
0,La(NO3)3,0.6445
1,Al(NO3)3,0.6271
2,La2(CO3)3,0.6092
3,Al(HO)3,0.6078
4,Ca(NO3)2,0.6032
5,Ce(NO3)3,0.6014
6,Sr(NO3)2,0.6011
7,Y(NO3)3,0.6001
8,La2O3,0.5980
9,LiNO3,0.5947


## 6 · Full precursor recommendation

The complete pipeline (= what the future MCP tool wraps): retrieval (§3) → borrow precursor sets →
MPC fills element gaps (§5) → optional reaction balancing (§7) filters unbalanceable sets.
`avoid` merges with the built-in unavailable-precursors list (`rsc/pres_unavail.json`).

In [7]:
# INPUT
targets = ["LiNi0.333Mn0.333Co0.333O2"]
top_n = 5
validate_reaction = True  # False: raw precursor sets, no balancing filter
avoid = []  # e.g. ["Co3O4"] to forbid a precursor

not_avail = rec.pre_set_unavail_default | set(avoid)
preds = rec.recommend_precursors(
    target_formula=targets,
    top_n=top_n,
    validate_reaction=validate_reaction,
    precursors_not_available=not_avail,
)
for tar, pred in zip(targets, preds):
    print("###", tar)
    if validate_reaction:
        for p in pred:
            print("  ", p["reaction_string"])
    else:
        pprint(pred)
    print()

len(test_targets_formulas) 1
top_n 5
x_index: 0 out of 1


### LiNi0.333Mn0.333Co0.333O2
   0.111 Co3O4 + 0.5 Li2CO3 + 0.333 MnO2 + 0.333 NiO + 0.028 O2 == 1 LiNi0.333Mn0.333Co0.333O2 + 0.5 CO2
   0.333 Co(CH3COO)2 + 1 LiOH + 0.333 Mn(CH3COO)2 + 0.333 Ni(CH3COO)2 + 0.25 O2 + 1.998 [OH-] == 1 LiNi0.333Mn0.333Co0.333O2 + 1.499 H2O + 1.998 [CH3COO-]



## 7 · Reaction balancing / validation

The non-ML safety net: can `target + precursors` balance into a stoichiometric reaction
(allowing volatile byproducts like CO2, H2O, O2, NH3)? Returns `None` when impossible —
such sets are silently dropped during recommendation when `validate_reaction=True`.
Test your own hypothetical routes here.

In [8]:
# INPUT
target = "BaTiO3"
precursors = ["BaCO3", "TiO2"]

rxn = rec.get_reaction(
    target_formula=target, precursors_formulas=precursors, ref_materials_comp={}
)
print(rxn[3] if rxn else "could not balance -> this set would be filtered out")

1 BaCO3 + 1 TiO2 == 1 BaTiO3 + 1 CO2


## 8 · Material decoder (the `mat` task)

The auxiliary training task: reconstruct the composition from the 32-dim embedding.
Good reconstruction = the embedding retains full composition information (it's not a lossy hash).
Two details worth noticing: the decoder has its **own element vocabulary** (82 elements — Tc is
excluded because it never occurs in text-mined targets), and `recover_mat` prints raw internal
tensors to the console — ignore that noise.

In [9]:
# INPUT
formula = "LiFePO4"

comp = formula_to_array(formula, ALL_ELES)
feats = featurize_list_of_composition(
    comps=[comp], ele_order=ALL_ELES, featurizer_type=rec.featurizer_type
)
decoded = model.recover_mat(np.array(feats)).numpy()[0]

# the decoder's own element vocabulary (excludes elements never seen in targets, e.g. Tc)
dec_eles = [l.decode() for l in model._model_by_task["mat"].target_ele_labels.numpy()]
true_frac = dict(zip(ALL_ELES, np.asarray(comp) / np.sum(comp)))
df = pd.DataFrame({"element": dec_eles, "decoded": decoded})
df["true fraction"] = df["element"].map(true_frac).fillna(0.0)
df[(df["true fraction"] > 0) | (df["decoded"] > 0.01)].sort_values(
    "decoded", ascending=False
).round(3)

[[-0.10414879 -0.140082642 -0.279814243 ... -0.234229729 0.0529021956 0.00584158907]] [[-0.10414879 -0.140082642 -0.279814243 ... -0.234229729 0.0529021956 0.00584158907]] [[-0.10414879 -0.140082642 -0.279814243 ... -0.234229729 0.0529021956 0.00584158907]]


,element,decoded,true fraction
80,O,0.568,0.571
62,P,0.159,0.143
6,Li,0.153,0.143
46,Fe,0.138,0.143


## Where to go next

- Change §3's target to something exotic (a doped composition) and follow how §6's recommendations
  trace back to the neighbors you saw.
- Read `recommend_precursors_by_similarity` in `scripts_utils/precursors_recommendation_utils.py`
  with §3 + §5 in mind — it is exactly those two cells plus bookkeeping.
- Benchmark logic: `scripts/_05_recommendation_benchmark.py::evaluation_prediction_precursors`
  scores §6-style outputs against §4-style ground truth.
- Extension template for new tasks (e.g. impurity prediction): `core/exp_models.py`.